# Imports

In [ ]:

# * for qwen
%pip install hf_transfer
%pip install accelerate
%pip install oauthlib
%pip install requests_oauthlib
%pip install torch
%pip install --upgrade transformers
%pip install matplotlib

In [ ]:
# import transformers
import requests
import torch

import PIL
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from collections import defaultdict

# for copernicus
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session
import io

import json

In [ ]:
# import cv2

# > for image to base64 conversion
import io
import base64

# > for getting API key securely
import getpass


In [ ]:

# temporary imports
import json
from PIL import Image
import os

# All sensor paths

In [ ]:
IGARSS_work_directory=os.getenv('IGARSS_work_directory')
IGARSS_work_directory=IGARSS_work_directory.replace('\\','/')

In [ ]:
sentinel_path = IGARSS_work_directory + "spectral_spatial_RSVQA_work_data/extracted_spectral_spatial_data/sentinel_2"

landsat_path = IGARSS_work_directory + "spectral_spatial_RSVQA_work_data/extracted_spectral_spatial_data/landsat"

lis3_path = IGARSS_work_directory + "spectral_spatial_RSVQA_work_data/extracted_spectral_spatial_data/liss_3"

lis4_path = IGARSS_work_directory + "spectral_spatial_RSVQA_work_data/extracted_spectral_spatial_data/liss_4"



# formatting functions

In [ ]:
class formatting_functions:
    def __init__(self):
        pass
    
    def format_vegetation_data(
        self,
        data_dict
        ):
        formatted_data_dict=dict()
        
        def format_cell_dict(cell_dict):
            formatted_cell_dict=dict()
            
            def format_area_coverage_info(area_coverage_info_dict):
                
                formatted_area_coverage_info_dict=dict()
                    
                for k,v in area_coverage_info_dict.items():
                        formatted_area_coverage_info_dict[k]=round(v,9)
                        
                return formatted_area_coverage_info_dict
            
            def format_ratio_coverage_info(ratio_coverage_info_dict):
                
                formatted_ratio_coverage_info_dict=dict()
                
                for k,v in ratio_coverage_info_dict.items():
                        formatted_ratio_coverage_info_dict[k]=round(v,2)
                        
                return formatted_ratio_coverage_info_dict
                    
            
            for k,v in cell_dict.items():
                
                if k=='total area covered by cell in km2':
                    formatted_cell_dict['total area covered by cell in km2']=round(v,9)
                    
                elif k=='area covered by cell':
                    formatted_cell_dict['area covered by cell km2']=round(v,9)
                
                
                elif k == "vegetation class coverage area in km2":
                    formatted_cell_dict[k]=v.copy()
                    
                    formatted_cell_dict[k]['coverage_info']=format_area_coverage_info(
                        v['coverage_info']
                        )
                    
                elif k == "vegetation class coverage %":
                    formatted_cell_dict[k]=v.copy()
                    
                    formatted_cell_dict[k]['coverage_info']=format_ratio_coverage_info(
                        v['coverage_info']
                        )

                elif k == "global morans I":
                    moran_i=v[0]
                    p_value = v[1]
                    formatted_cell_dict[k]={
                        "moran's I":moran_i,
                        "p-value":p_value
                    }

                else: formatted_cell_dict[k]=v.copy()
                
            return formatted_cell_dict
                
        
        cell_map={
            'cell_1':"upper left",
            'cell_2':"upper middle",
            'cell_3':"upper right",
            'cell_4':"middle left",
            'cell_5':"center",
            'cell_6':"middle right",
            'cell_7':"lower left",
            'cell_8':"lower middle",
            'cell_9':"lower right"}
        
        if len(data_dict)>0:
            for cell, data in data_dict.items():
                formatted_data_dict[cell_map[cell]] = format_cell_dict(data)
            
        return formatted_data_dict
    
    def format_built_data(
        self,
        data_dict
        ):
        formatted_data_dict=dict()
        
        def format_cell_dict(cell_dict):
            formatted_cell_dict=dict()
            
            def format_area_coverage_info(area_coverage_info_dict):
                
                formatted_area_coverage_info_dict=dict()
                    
                for k,v in area_coverage_info_dict.items():
                        formatted_area_coverage_info_dict[k]=round(v,9)
                        
                return formatted_area_coverage_info_dict
            
            def format_ratio_coverage_info(ratio_coverage_info_dict):
                
                formatted_ratio_coverage_info_dict=dict()
                
                for k,v in ratio_coverage_info_dict.items():
                        formatted_ratio_coverage_info_dict[k]=round(v,2)
                        
                return formatted_ratio_coverage_info_dict
                    
            
            for k,v in cell_dict.items():
                
                
                if k == "built-up class coverage %":
                    
                    formatted_cell_dict[k]={}
                    
                    for k2,v2 in v.items():
                        if k2=="coverage":
                            formatted_cell_dict[k]["coverage_info"]=v2.copy()
                            
                        else:
                            formatted_cell_dict[k][k2]=v2
                            
                    
                    formatted_cell_dict[k]['coverage_info']=format_ratio_coverage_info(
                        v['coverage']
                        )
                    
                else:
                    formatted_cell_dict[k]=v
                
            return formatted_cell_dict
                
        
        cell_map={
            'cell_1':"upper left",
            'cell_2':"upper middle",
            'cell_3':"upper right",
            'cell_4':"middle left",
            'cell_5':"center",
            'cell_6':"middle right",
            'cell_7':"lower left",
            'cell_8':"lower middle",
            'cell_9':"lower right"}
        
        if len(data_dict)>0:
            for cell, data in data_dict.items():
                formatted_data_dict[cell_map[cell]] = format_cell_dict(data)
            
        return formatted_data_dict
    
    def format_overall_data(
        self,
        data_dict
        ):
        
        formatted_data_dict=dict()
        
        if len(data_dict)>0:
            for k,v in data_dict.items():
                if k=='total area in km2':
                    formatted_data_dict[k]=round(v,9)
                else:
                    formatted_data_dict[k]=v.copy()
                
        return formatted_data_dict
    
    def format_water_data(
        self,
        data_dict
        ):
        formatted_data_dict=dict()
        
        if len(data_dict)>0:
            for k,v in data_dict.items():
                
                formatted_v=dict()
                formatted_v["bounding box"]=v["bounding box"].copy()

                formatted_v["area in m2"]=round(v["area in m2"],3)
                
                formatted_data_dict[f"water body {k}"]=formatted_v
            
        return formatted_data_dict

# function for resizing image

In [ ]:
from PIL import Image
import math

def resize_image_to_limit(high_res_img, max_pixels=33177600):
    
    import math
    
    width, height = high_res_img.size
    current_pixels = width * height

    # If already within limit, just save and return
    if current_pixels <= max_pixels:
        return high_res_img

    # Compute scale factor
    scale_factor = math.sqrt(max_pixels / current_pixels)

    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)

    print(
        f"Resizing image from ({width}x{height}={current_pixels} px) "
        f"→ ({new_width}x{new_height}={new_width*new_height} px)"
    )

    # Resize with high quality
    resized_img = high_res_img.resize((new_width, new_height), Image.LANCZOS)
    
    return resized_img


# function for extracting json data

In [ ]:

def get_data(
    bands_dir_path:str,
    sensor:str
):
    
    formatter=formatting_functions()
    info_file_key_words=[]
        
    if sensor!='lis4':
        info_file_key_words=[
            "water",
            "built",
            "veg",
            "overall",
            "location",
            "bounds",
            "spatial"
            ]
    else:
        info_file_key_words=[
            "water",
            "veg",
            "overall",
            "location",
            "bounds",
            "spatial"
            ]
    
    paths_dict=dict()
    
    for file_key_word in info_file_key_words:
        
        paths_dict[f"{file_key_word}_info_path"] = [
            os.path.join(
                bands_dir_path,
                file_name) for file_name in os.listdir(bands_dir_path)
            if (file_key_word in file_name) and (file_name.endswith(".json"))
            ][0]
    
    
    data_dict=dict()
    
    for path_name, path in paths_dict.items():
        
        file_key_word=path_name.split("_")[0]
        data_name_key=f"{file_key_word}_data"
        
        try:
            with open(path, "r", encoding="utf-8") as f:
                
                json_data=json.load(f)

                if file_key_word=="water":
                    data_dict[data_name_key]=formatter.format_water_data(json_data)
                    
                elif file_key_word=="built":
                    data_dict[data_name_key]=formatter.format_built_data(json_data)
                    
                elif file_key_word=="veg":
                    data_dict[data_name_key]=formatter.format_vegetation_data(json_data)
                    
                elif file_key_word=="overall":
                    data_dict[data_name_key]=formatter.format_overall_data(json_data)

                elif file_key_word=="spatial":
                    data_dict[data_name_key]=json_data

                else:
                    data_dict[data_name_key]=json_data
                
        except FileNotFoundError:
            print(f"{path} file not found!")
    
    # * open the image
    image_path=[
            os.path.join(
                bands_dir_path,
                file_name) for file_name in os.listdir(bands_dir_path)
            if file_name.endswith(".jpg")
            ][0]
    
    
    img = Image.open(image_path)
            
    return data_dict,img

# Prompt

## Without context prompt

### model instructions without context with output instructions

In [ ]:
ModelInstructions_without_context_with_OutputInstructions="""
# Identity

You are specifically designed to address queries related to satellite imagery.
You can generate human-like text based on the input you receive, enabling natural and coherent conversations that remain relevant to the topic at hand. You are capable of understanding large amounts of information from remote sensing images and related knowledge.

As a specialized language model, you are limited to interpreting remote sensing images and answering questions about them using only the visual features visible in the image.

Since image filenames may be arbitrary, you must not rely on them when forming answers.

You remain faithful to the actual features visible in the remote sensing image rather than fabricating content.

Overall, you serve as a powerful visual dialogue assistant for remote sensing tasks, providing insights extracted strictly from satellite imagery.

# Instructions

* Use only observable visual evidence.
* Output ONLY a valid JSON object.
* Never add contextual background, methodology, assumptions, or uncertainty.
* The JSON object MUST conform exactly to the output schema.
* If the answer cannot be determined, return null for the value.
"""


### user text input without context with output format

In [ ]:
UserTextInput_without_context_with_OutputFormat="""
**Spatial metadata:**

*(Note: Provides technical spatial reference details only. This may include the coordinate reference system (CRS), coordinate type (geographic or projected), unit of measurement, pixel resolution, and pixel ground area if explicitly available. Do not infer missing values.)*


**question:**

{question}

**output schema:**

```json
{output_schema}
```
"""

### user text input without context with ouput format with example

In [ ]:
UserTextInput_without_context_with_OutputFormat_with_example="""
**Spatial metadata:**

*(Note: Provides technical spatial reference details only. This may include the coordinate reference system (CRS), coordinate type (geographic or projected), unit of measurement, pixel resolution, and pixel ground area if explicitly available. Do not infer missing values.)*

{spatial_metadata}

**question:**

{question}

**output schema:**

```json
{output_schema}
```

**example response:**

question: "What is the area covered by the water body?"

expected response:

```json
{{
  'answer': 723.32,
  'unit': 'm2'
}}
```
"""

### model instructions without context without output instructions

In [ ]:
ModelInstructions_without_context_without_OutputInstructions="""
# Identity

You are specifically designed to address queries related to satellite imagery.
You can generate human-like text based on the input you receive, enabling natural and coherent conversations that remain relevant to the topic at hand. You are capable of understanding large amounts of information from remote sensing images and related knowledge.

As a specialized language model, you are limited to interpreting remote sensing images and answering questions about them using only the visual features visible in the image.

Since image filenames may be arbitrary, you must not rely on them when forming answers.

You remain faithful to the actual features visible in the remote sensing image rather than fabricating content.

Overall, you serve as a powerful visual dialogue assistant for remote sensing tasks, providing insights extracted strictly from satellite imagery.

# Instructions

* Use only observable visual evidence.
* Never add contextual background, methodology, assumptions, or uncertainty.
* If the answer cannot be determined, return null for the value.
"""


### user text input without context without output format

In [ ]:
UserTextInput_without_context_without_OutputFormat="""
**Spatial metadata:**

*(Note: Provides technical spatial reference details only. This may include the coordinate reference system (CRS), coordinate type (geographic or projected), unit of measurement, pixel resolution, and pixel ground area if explicitly available. Do not infer missing values.)*

{spatial_metadata}

**question:**

{question}
"""

## with context prompt

### model instructions with context with output instructions

In [ ]:
ModelInstructions_with_context_with_OutputInstructions="""
# Identity

You are specifically designed to address queries related to satellite imagery.
You can generate human-like text based on the input you receive, enabling natural and coherent conversations that remain relevant to the topic at hand. You are capable of understanding large amounts of information from remote sensing images, related knowledge, and structured land-cover information.

As a specialized language model, you are limited to interpreting remote sensing images and answering questions about them using only:
(1) the visual features visible in the image and
(2) the supplementary information provided below.

Since image filenames may be arbitrary, you must not rely on them when forming answers.

You remain faithful to the actual features visible in the remote sensing image and the supplementary information, rather than fabricating content.

When supplementary information is present, you must incorporate it into your answers and prefer it over visual inference when necessary.

Overall, you serve as a powerful visual dialogue assistant for remote sensing tasks, providing insights extracted strictly from satellite imagery and supplied information.

# Instructions

* Use only observable visual evidence.
* Output ONLY a valid JSON object.
* Never add contextual background, methodology, assumptions, or uncertainty.
* The JSON object MUST conform exactly to the output schema.
* If the answer cannot be determined, return null for the value.

* **Always reference either visual evidence or specific supplementary data in your sentence.**
"""

### user text input with context with output format

In [ ]:
UserTextInput_with_context_with_OutputFormat="""
# Provided Information

**Spatial metadata:**

*(Note: Provides technical spatial reference details only. This may include the coordinate reference system (CRS), coordinate type (geographic or projected), unit of measurement, pixel resolution, and pixel ground area if explicitly available. Do not infer missing values.)*

{spatial_metadata}

**Image bound info:**

*(Note: Provides the exact latitude and longitude coordinates (in decimal degrees) defining the geographic extent of the satellite image.)*

{image_bounds}

**Location info:**

*(Note: Indicates the ISO country code, country name, and the primary city corresponding to the area covered by the image.)*

{location_info}

**Water bodies info:**

*(Note: Lists each detected water body's bounding box coordinates and the surface area of each in square meters (m2).)*

{water_body_info}

**Vegetation info:**

*(Note: The image is divided into 9 cells; for each cell, following data is given: (1) total cell area (km2), (2) percentage (%) of that cell's area and corresponding area (km2) classified by vegetation density (no vegetation, sparse/bare ground, moderate, dense), and (3) Moran's I value indicating spatial clustering of vegetation.)*

{vegetation_info}

**Built-up info:**

*(Note: The image is divided into 9 cells; for each cell, following data is given: (1) total cell area (km2), and (2) percentage (%) of that cell's area classified as non-built, moderate built-up, and dense built-up.)*

{built_info}

**Overall land cover info:**

*(Note: This section reports: (1) the total area of the satellite image (km2), (2) the percentage (%) of the entire image covered by each land cover category (unclassified land, water, moderate vegetation, dense vegetation, moderate built-up, dense built-up), and (3) the corresponding area (km2) for each category.)*

{overall_land_stat}

**question:**

{question}

**output schema:**

{output_schema}
"""

### user text input with context with output format with example

In [ ]:
UserTextInput_with_context_with_OutputFormat_with_example="""
# Provided Information

**Spatial metadata:**

*(Note: Provides technical spatial reference details only. This may include the coordinate reference system (CRS), coordinate type (geographic or projected), unit of measurement, pixel resolution, and pixel ground area if explicitly available. Do not infer missing values.)*

{spatial_metadata}

**Image bound info:**

*(Note: Provides the exact latitude and longitude coordinates (in decimal degrees) defining the geographic extent of the satellite image.)*

{image_bounds}

**Location info:**

*(Note: Indicates the ISO country code, country name, and the primary city corresponding to the area covered by the image.)*

{location_info}

**Water bodies info:**

*(Note: Lists each detected water body's bounding box coordinates and the surface area of each in square meters (m2).)*

{water_body_info}

**Vegetation info:**

*(Note: The image is divided into 9 cells; for each cell, following data is given: (1) total cell area (km2), (2) percentage (%) of that cell's area and corresponding area (km2) classified by vegetation density (no vegetation, sparse/bare ground, moderate, dense), and (3) Moran's I value indicating spatial clustering of vegetation.)*

{vegetation_info}

**Built-up info:**

*(Note: The image is divided into 9 cells; for each cell, following data is given: (1) total cell area (km2), and (2) percentage (%) of that cell's area classified as non-built, moderate built-up, and dense built-up.)*

{built_info}

**Overall land cover info:**

*(Note: This section reports: (1) the total area of the satellite image (km2), (2) the percentage (%) of the entire image covered by each land cover category (unclassified land, water, moderate vegetation, dense vegetation, moderate built-up, dense built-up), and (3) the corresponding area (km2) for each category.)*

{overall_land_stat}

**question:**

{question}

**output schema:**

{output_schema}

**example response:**

question: "What is the area covered by the water body?"

expected response:

```json
{{
  'answer': 64.32,
  'unit': 'm2'
}}
```
"""

### model instructions with context without output instructions

In [ ]:
ModelInstructions_with_context_without_OutputInstructions="""
# Identity

You are specifically designed to address queries related to satellite imagery.
You can generate human-like text based on the input you receive, enabling natural and coherent conversations that remain relevant to the topic at hand. You are capable of understanding large amounts of information from remote sensing images, related knowledge, and structured land-cover information.

As a specialized language model, you are limited to interpreting remote sensing images and answering questions about them using only:
(1) the visual features visible in the image and
(2) the supplementary information provided below.

Since image filenames may be arbitrary, you must not rely on them when forming answers.

You remain faithful to the actual features visible in the remote sensing image and the supplementary information, rather than fabricating content.

When supplementary information is present, you must incorporate it into your answers and prefer it over visual inference when necessary.

Overall, you serve as a powerful visual dialogue assistant for remote sensing tasks, providing insights extracted strictly from satellite imagery and supplied information.

# Instructions

* Never add contextual background, methodology, assumptions, or uncertainty.
* If the answer cannot be determined, return null for the value.

* **Always reference either visual evidence or specific supplementary data in your sentence.**
"""

### user text input with context without output format

In [ ]:
UserTextInput_with_context_without_OutputFormat="""
# Provided Information

**Spatial metadata:**

*(Note: Provides technical spatial reference details only. This may include the coordinate reference system (CRS), coordinate type (geographic or projected), unit of measurement, pixel resolution, and pixel ground area if explicitly available. Do not infer missing values.)*

{spatial_metadata}

**Image bound info:**

*(Note: Provides the exact latitude and longitude coordinates (in decimal degrees) defining the geographic extent of the satellite image.)*

{image_bounds}

**Location info:**

*(Note: Indicates the ISO country code, country name, and the primary city corresponding to the area covered by the image.)*

{location_info}

**Water bodies info:**

*(Note: Lists each detected water body's bounding box coordinates and the surface area of each in square meters (m2).)*

{water_body_info}

**Vegetation info:**

*(Note: The image is divided into 9 cells; for each cell, following data is given: (1) total cell area (km2), (2) percentage (%) of that cell's area and corresponding area (km2) classified by vegetation density (no vegetation, sparse/bare ground, moderate, dense), and (3) Moran's I value indicating spatial clustering of vegetation.)*

{vegetation_info}

**Built-up info:**

*(Note: The image is divided into 9 cells; for each cell, following data is given: (1) total cell area (km2), and (2) percentage (%) of that cell's area classified as non-built, moderate built-up, and dense built-up.)*

{built_info}

**Overall land cover info:**

*(Note: This section reports: (1) the total area of the satellite image (km2), (2) the percentage (%) of the entire image covered by each land cover category (unclassified land, water, moderate vegetation, dense vegetation, moderate built-up, dense built-up), and (3) the corresponding area (km2) for each category.)*

{overall_land_stat}

**question:**

{question}
"""

# questions

In [ ]:
ques_v2_dict = {

    "object_counting":[
        "How many distinct water bodies are present (give only a number)?",
        "How many distinct land cover classes are present (give only a number)?",],

    "object_area": [
        # water
        "What is the total water body area (give one word answer in km2)?",
        "What percentage of total area is water (give one word answer in %)?",


        # vegetation
        "What is the total vegetation area (give one word answer in km2)?",
        "What percentage of area is vegetated (give one word answer in %)?",


        # built-up
        "What is the total built-up area (give one word answer in km2)?",
        "What percentage of area is built-up (give one word answer in %)?",


    ],

    "object_localization": [
        "Where is the most vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Where is the least vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Where is the most dense vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Where is the most sparse vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Which area has the most fragmented vegetation cover (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Which area has most continuous vegetation cover (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Where is the most built-up in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Where is the least built-up in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Where is the most dense built-up in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?",
        "Give location of the largest water body.",
        "Give location of the smallest water body."
    ],

    "object_presence": [
        "Is water present in the image (yes/no)?",
        "Is vegetation present (yes/no)?",
        "Is dense vegetation present (yes/no)?",
        "Is built-up present in the image (yes/no)?",

        # vegetation presence per grid
        "Is vegetation present in upper left (yes/no)?",
        "Is vegetation present in upper middle (yes/no)?",
        "Is vegetation present in upper right (yes/no)?",
        "Is vegetation present in middle left (yes/no)?",
        "Is vegetation present in center (yes/no)?",
        "Is vegetation present in middle right (yes/no)?",
        "Is vegetation present in lower left (yes/no)?",
        "Is vegetation present in lower middle (yes/no)?",
        "Is vegetation present in lower right (yes/no)?",

        # dense vegetation presence per grid
        "Is dense vegetation present in upper left (yes/no)?",
        "Is dense vegetation present in upper middle (yes/no)?",
        "Is dense vegetation present in upper right (yes/no)?",
        "Is dense vegetation present in middle left (yes/no)?",
        "Is dense vegetation present in center (yes/no)?",
        "Is dense vegetation present in middle right (yes/no)?",
        "Is dense vegetation present in lower left (yes/no)?",
        "Is dense vegetation present in lower middle (yes/no)?",
        "Is dense vegetation present in lower right (yes/no)?",

        # built-up presence per grid
        "Is built-up present in upper left (yes/no)?",
        "Is built-up present in upper middle (yes/no)?",
        "Is built-up present in upper right (yes/no)?",
        "Is built-up present in middle left (yes/no)?",
        "Is built-up present in center (yes/no)?",
        "Is built-up present in middle right (yes/no)?",
        "Is built-up present in lower left (yes/no)?",
        "Is built-up present in lower middle (yes/no)?",
        "Is built-up present in lower right (yes/no)?",

        # dense built-up presence per grid
        "Is dense built-up present in upper left (yes/no)?",
        "Is dense built-up present in upper middle (yes/no)?",
        "Is dense built-up present in upper right (yes/no)?",
        "Is dense built-up present in middle left (yes/no)?",
        "Is dense built-up present in center (yes/no)?",
        "Is dense built-up present in middle right (yes/no)?",
        "Is dense built-up present in lower left (yes/no)?",
        "Is dense built-up present in lower middle (yes/no)?",
        "Is dense built-up present in lower right (yes/no)?",

    ],

    "attribute_recognition": [
        "Is the majority vegetation healthy or stressed (choose one from 'healthy', 'stressed')?",
        "Is the majority built-up low, medium, or high density (choose one from 'low density', 'medium density', 'high density')?",
        "Is the majority  vegetation sparse, moderate, or dense (choose one from 'sparse', 'moderate', 'dense')?",
        "What is the land cover dominance (answer in one word)?",
        "What land cover is rare (answer in one word)?",
        "Which class is more dominating in the image ('water', 'built-up', 'vegetation')?"
    ],

    "image_caption": [
        'What does the image show? Summarize the content of the image.',
        'List 3 most prominent landscape features (e.g., river, forest patch, urban area).',
        'Give an overall summary of built-up structures in the image.',
        'Give the overall summary of water bodies in the image'
    ],

    "attribute_reasoning":[
        "Provide an integrated summary describing vegetation, water and built-up, including key numbers (percentages/areas)."
    ]
}


## all questions list

In [ ]:
list_of_question_answer_list = ques_v2_dict.values()

all_ques_list=[]
for question_list in list_of_question_answer_list:
    all_ques_list.extend(question_list)

all_ques_list

## lis4 questions

In [ ]:
lis4_ques_list = [ques for ques in all_ques_list if "built" not in ques]
lis4_ques_list

In [ ]:
set(all_ques_list) - set(lis4_ques_list)

## Caption ans attribute reasoning questions

In [ ]:
caption_and_attribute_reasoning_ques=[
    'What does the image show? Summarize the content of the image.',
    'List 3 most prominent landscape features (e.g., river, forest patch, urban area).',
    'Give an overall summary of built-up structures in the image.',
    'Give the overall summary of water bodies in the image'
    "Provide an integrated summary describing vegetation, water and built-up, including key numbers (percentages/areas)."]

In [ ]:

# object presence
object_presence_output_schema = {
    "type": "object",
    "description": "Answer for object presence question.",
    "properties": {
        "answer": {
            "type": "boolean",
            "description": "true if present, false otherwise"
        },
    },
    "required": ["answer"]
}

# object_counting_schema
object_counting_output_schema = {
    "type": "object",
    "description": "Answer for object counting question.",
    "properties": {
        "answer": {
            "type": "integer",
            "description": "Integer count."
        },
    },
    "required": ["answer"]
}

object_area_output_schema = {
    "type": "object",
    "description": "Answer for object area calculation question.",
    "properties": {
        "answer": {
            "type": "number",
            "description": "Numerical answer to the question."},
        "unit": {
            "type": "string",
            "description": "Unit of the numerical answer, e.g., 'km2' or 'm2' or '%'."
        }
    },
    "required": ["answer"]
}

object_localization_output_schema = {
    "type": "object",
    "description": "Answer for object localization question.",
    "properties": {
        "answer": {
            "type": "string",
            "description": "Location answer to the question.",
            "enum":[
                'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right'
            ]
        },
    },
    "required": ["answer"]
}

attribute_recognition_output_schema = {
    "type": "object",
    "description": "Answer for attribute recognition question.",
    "properties": {
        "answer": {
            "type": "string",
            "description": "One word answer to the question."
        },
    },
    "required": ["answer"]
}

caption_output_schema = {
    "type": "object",
    "description": "Answer for caption question.",
    "properties": {
        "answer": {
            "type": "string",
            "description": "Textual answer to the question."
        },
    },
    "required": ["answer"]
}

attribute_reasoning_output_schema = {
    "type": "object",
    "description": "Answer for attribute reasoning question.",
    "properties": {
        "answer": {
            "type": "string",
            "description": "Textual answer to the question."
        },
    },
    "required": ["answer"]
}

# Defining output message format

# make_message function and
# message_template

In [ ]:
# > for showing message template in output file
message_template= """[
    {
        "role": "developer",
        "content": [
            {
                "type": "text", "text": 'prompt'
            }
            ],
    },
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": 'img'
            },
            {
                "type": "text", "text": "Question: {query}"
            }
            ],
    },
    {
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Therefore the answer is"
            }
            ],
    },

]"""


# *=================================================================================================================================================================


# > function for making message
def make_message(
        image,
        model_instructions:str,
        user_text_input:str
        ):
    messages = [
        {
            "role": "developer",
            "content": [
                {
                    "type": "text",
                    "text": model_instructions

                }
                ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image":image,
                },
                {
                    "type": "text",
                    "text": user_text_input
                }
                ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",

                }
                ],
        },

    ]

    return messages


# Load full model

In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate


In [ ]:

!pip install qwen-vl-utils[decord]==0.0.8


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "AdaptLLM/remote-sensing-Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto",
    token="your_hf_token"
)

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "AdaptLLM/remote-sensing-Qwen2.5-VL-3B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("AdaptLLM/remote-sensing-Qwen2.5-VL-3B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("AdaptLLM/remote-sensing-Qwen2.5-VL-3B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


# run_for_sensor function

In [ ]:
import copy

In [ ]:
def run_for_sensor(
        model_name:str,
        sensor:str,
        parent_dir_path:str,
        questions,
        max_new_tokens:int,

        ch_ModelInstructions_with_context_with_OutputInstructions:bool,
        ch_ModelInstructions_with_context_without_OutputInstructions:bool,

        ch_ModelInstructions_without_context_with_OutputInstructions:bool,
        ch_ModelInstructions_without_context_without_OutputInstructions:bool,

        ch_UserTextInput_with_context_without_OutputFormat:bool,
        ch_UserTextInput_with_context_with_OutputFormat:bool,
        ch_UserTextInput_with_context_with_OutputFormat_with_example:bool,

        ch_UserTextInput_without_context_without_OutputFormat:bool,
        ch_UserTextInput_without_context_with_OutputFormat:bool,
        ch_UserTextInput_without_context_with_OutputFormat_with_example:bool

):
    # ====================================================================================================================================
    def save_in_json(
            content:dict,
            path:str,
            file_name:str
            ):
            """Saves a dictionary as json file.

            Args:
                content (dict): Dictionary containing the content
                path (str): The directory in which json file will be saved, for eg. "C/abc/.../sentinel/all_bands"
                file_name (str): Name of the json file, for eg. "water_info.json"
            """

            with open(os.path.join(path,file_name), "w") as f:
                json.dump(content, f)

    def format_UserTextInput_with_context_without_OutputFormat(
            UserTextInput,
            json_data,
            question
    ):
        return UserTextInput.format(
            question=question,
            vegetation_info = json_data['veg_data'],
            built_info = json_data['built_data'] if sensor!='lis4' else None,
            water_body_info = json_data['water_data'],
            overall_land_stat = json_data['overall_data'],
            location_info = json_data['location_data'],
            image_bounds = json_data['bounds_data'],
            spatial_metadata = json_data['spatial_data'],
        )

    def format_UserTextInput_with_context_with_OutputFormat(
            UserTextInput,
            json_data,
            question
    ):
        output_schema=None

        if question in ques_v2_dict['object_presence']:
            output_schema=object_presence_output_schema

        elif question in ques_v2_dict['object_counting']:
            output_schema=object_counting_output_schema

        elif question in ques_v2_dict['object_area']:
            output_schema=object_area_output_schema

        elif question in ques_v2_dict['object_localization']:
            output_schema=object_localization_output_schema

        elif question in ques_v2_dict['attribute_recognition']:
            output_schema=attribute_recognition_output_schema

        elif question in ques_v2_dict['image_caption']:
            output_schema=caption_output_schema

        elif question in ques_v2_dict['attribute_reasoning']:
            output_schema=attribute_reasoning_output_schema

            if "choose one from" in question:
                demo_q=question
                output_schema['properties']['answer']['enum']=demo_q.replace("'","").split("(choose one from ")[-1].split(")?")[0].split(",")

        else:
            raise ValueError("Question not found in any category.")

        return UserTextInput.format(
            question=question,
            output_schema=output_schema,

            vegetation_info = json_data['veg_data'],
            built_info = json_data['built_data'] if sensor!='lis4' else None,
            water_body_info = json_data['water_data'],
            overall_land_stat = json_data['overall_data'],
            location_info = json_data['location_data'],
            image_bounds = json_data['bounds_data'],
            spatial_metadata = json_data['spatial_data'],
        )

    def format_UserTextInput_without_context_with_OutputFormat(
            UserTextInput,
            question
    ):
        output_schema=None

        if question in ques_v2_dict['object_presence']:
            output_schema=object_presence_output_schema

        elif question in ques_v2_dict['object_counting']:
            output_schema=object_counting_output_schema

        elif question in ques_v2_dict['object_area']:
            output_schema=object_area_output_schema

        elif question in ques_v2_dict['object_localization']:
            output_schema=object_localization_output_schema

        elif question in ques_v2_dict['attribute_recognition']:
            output_schema=attribute_recognition_output_schema

        elif question in ques_v2_dict['image_caption']:
            output_schema=caption_output_schema

        elif question in ques_v2_dict['attribute_reasoning']:
            output_schema=attribute_reasoning_output_schema

            if "choose one from" in question:
                demo_q=question
                output_schema['properties']['answer']['enum']=demo_q.replace("'","").split("(choose one from ")[-1].split(")?")[0].split(",")

        else:
            raise ValueError("Question not found in any category.")

        return UserTextInput.format(
            spatial_metadata = json_data['spatial_data'],
            question=question,
            output_schema=output_schema,
        )

    # ====================================================================================================================================

    if sum([
        ch_ModelInstructions_with_context_with_OutputInstructions,
        ch_ModelInstructions_with_context_without_OutputInstructions,
        ch_ModelInstructions_without_context_with_OutputInstructions,
        ch_ModelInstructions_without_context_without_OutputInstructions
    ])!=1:
        raise ValueError("Only one of the ModelInstructions options must be True.")
    if sum([
        ch_UserTextInput_with_context_with_OutputFormat,
        ch_UserTextInput_with_context_with_OutputFormat_with_example,
        ch_UserTextInput_with_context_without_OutputFormat,
        ch_UserTextInput_without_context_with_OutputFormat,
        ch_UserTextInput_without_context_with_OutputFormat_with_example,
        ch_UserTextInput_without_context_without_OutputFormat,

    ])!=1:
        raise ValueError("Only one of the UserTextInput options must be True.")

    if ch_ModelInstructions_without_context_without_OutputInstructions or ch_ModelInstructions_without_context_with_OutputInstructions:
        if ch_UserTextInput_with_context_without_OutputFormat or ch_UserTextInput_with_context_with_OutputFormat:
            raise ValueError("Context cannot be present in UserTextInput when ModelInstructions does not have context.")

    if ch_ModelInstructions_with_context_without_OutputInstructions or ch_ModelInstructions_with_context_with_OutputInstructions:
        if ch_UserTextInput_without_context_without_OutputFormat or ch_UserTextInput_without_context_with_OutputFormat:
            raise ValueError("Context must be present in UserTextInput when ModelInstructions has context.")

    output_file_name=""
    if any([ch_ModelInstructions_with_context_with_OutputInstructions, ch_ModelInstructions_with_context_without_OutputInstructions]):
        output_file_name=f"fine_tuned_{model_name}_with_metadata_output.json"

    else:
        output_file_name=f"fine_tuned_{model_name}_output.json"


    sensor_ques_ans_dict=dict()
    unsaved_aoi_ques_ans_dict=dict()
    updated_aoi_ques_ans_dict=dict()

    bands_dir_names=os.listdir(parent_dir_path)

    for i,band_dir_name in enumerate(bands_dir_names, start=0):

        print(i," : ",band_dir_name)
        bands_dir_path=os.path.join(
            parent_dir_path,
            band_dir_name
        )

        saved_ans_json_data={}

        # * if output_file_name already exists, then open it
        if f"{output_file_name}" in os.listdir(bands_dir_path):
            print("\n!!!!!!!!! generated_answers.json already exists, hence will be updated !!!!!!!!!\n")


            # * open answers json file

            ans_file_name=f"{output_file_name}"
            ans_file_path=os.path.join(
                bands_dir_path,
                ans_file_name
            )

            with open(ans_file_path, "r", encoding="utf-8") as f:
                saved_ans_json_data=json.load(f)


        # * if output_file_name not present, then make it
        else:
            save_in_json(
                content=saved_ans_json_data,
                path=bands_dir_path,
                file_name=f"{output_file_name}"
            )


        json_data,jpg_img=get_data(
            bands_dir_path=bands_dir_path,
            sensor=sensor
        )

        # * run for only unsaved questions
        unsaved_questions=questions
        if len(saved_ans_json_data)!=0:

            saved_questions = list(saved_ans_json_data.keys())
            unsaved_questions = list( set(questions) - set( saved_questions ) )
            print("Already saved questions:-\n")
            for q in saved_questions:
                print(q)

            print("==========================================================\n")
            print("unsaved questions:-\n")
            for q in unsaved_questions:
                print(q)

        if len(unsaved_questions)==0:
            print("!!!!!!!!! to questions to save !!!!!!!!!")
            return
        
        for q in unsaved_questions:
            print('- ',q)


            final_model_instructions=""
            final_user_text_input=""

            if ch_ModelInstructions_with_context_with_OutputInstructions:
                final_model_instructions = ModelInstructions_with_context_with_OutputInstructions
            elif ch_ModelInstructions_with_context_without_OutputInstructions:
                final_model_instructions = ModelInstructions_with_context_without_OutputInstructions
            elif ch_ModelInstructions_without_context_with_OutputInstructions:
                final_model_instructions = ModelInstructions_without_context_with_OutputInstructions
            elif ch_ModelInstructions_without_context_without_OutputInstructions:
                final_model_instructions = ModelInstructions_without_context_without_OutputInstructions

            if ch_UserTextInput_with_context_with_OutputFormat:
                final_user_text_input = format_UserTextInput_with_context_with_OutputFormat(
                    UserTextInput=UserTextInput_with_context_with_OutputFormat,
                    json_data=json_data,
                    question=q
                )
            elif ch_UserTextInput_with_context_with_OutputFormat_with_example:
                final_user_text_input = format_UserTextInput_with_context_with_OutputFormat(
                    UserTextInput=UserTextInput_with_context_with_OutputFormat_with_example,
                    json_data=json_data,
                    question=q
                )
            elif ch_UserTextInput_with_context_without_OutputFormat:
                final_user_text_input = format_UserTextInput_with_context_without_OutputFormat(
                    UserTextInput=UserTextInput_with_context_without_OutputFormat,
                    json_data=json_data,
                    question=q
                )
            elif ch_UserTextInput_without_context_with_OutputFormat:
                final_user_text_input = format_UserTextInput_without_context_with_OutputFormat(
                    UserTextInput=UserTextInput_without_context_with_OutputFormat,
                    question=q
                )
            elif ch_UserTextInput_without_context_with_OutputFormat_with_example:
                final_user_text_input = format_UserTextInput_without_context_with_OutputFormat(
                    UserTextInput=UserTextInput_without_context_with_OutputFormat_with_example,
                    question=q
                )

            elif ch_UserTextInput_without_context_without_OutputFormat:
                final_user_text_input = UserTextInput_without_context_without_OutputFormat.format(question=q)
        
            # ==========================================================================================
            # ==========================================================================================
            
            messages=make_message(
                image=jpg_img,
                model_instructions=final_model_instructions,
                user_text_input= final_user_text_input
                )

            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True)
            
            image_inputs, video_inputs = process_vision_info(messages)
            
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to("cuda")

            # Inference: Generation of the output
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens)
            
            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]
            
            output_text = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )
            
            # ==========================================================================================
            # ==========================================================================================
            unsaved_aoi_ques_ans_dict[q] = output_text

            print(">",output_text)

            updated_aoi_ques_ans_dict = unsaved_aoi_ques_ans_dict.copy()

            # * add new question, answers from unsaved_aoi_ques_ans_dict to updated_aoi_ques_ans_dict
            if len(saved_ans_json_data)!=0:

                updated_aoi_ques_ans_dict = saved_ans_json_data.copy()

                for new_q, new_a in unsaved_aoi_ques_ans_dict.items():
                    updated_aoi_ques_ans_dict[new_q]=new_a


            # * save as json in the directory
            save_in_json(
                content=updated_aoi_ques_ans_dict,
                path=bands_dir_path,
                file_name=f"{output_file_name}"
            )


        sensor_ques_ans_dict[band_dir_name] = copy.deepcopy(updated_aoi_ques_ans_dict)




    return sensor_ques_ans_dict


# Running for sensors

## landsat

### without context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="landsat",
    parent_dir_path=landsat_path,
    questions=all_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=False,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=True,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=False,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=True,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

### with context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="landsat",
    parent_dir_path=landsat_path,
    questions=all_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=True,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=False,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=True,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

## lis3

### without context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="lis3",
    parent_dir_path=lis3_path,
    questions=all_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=False,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=True,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=False,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=True,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

### with context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="lis3",
    parent_dir_path=lis3_path,
    questions=all_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=True,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=False,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=True,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

## sentinel

### without context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="sentinel",
    parent_dir_path=sentinel_path,
    questions=all_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=False,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=True,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=False,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=True,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

### with context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="sentinel",
    parent_dir_path=sentinel_path,
    questions=all_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=True,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=False,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=True,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

## lis4

### without context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="lis4",
    parent_dir_path=lis4_path,
    questions=lis4_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=False,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=True,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=False,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=True,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)

### with context

In [ ]:
ques_ans=run_for_sensor(
    model_name="qwen",
    sensor="lis4",
    parent_dir_path=lis4_path,
    questions=lis4_ques_list,
    max_new_tokens=260,
    ch_ModelInstructions_with_context_with_OutputInstructions=True,
    ch_ModelInstructions_with_context_without_OutputInstructions=False,
    ch_ModelInstructions_without_context_with_OutputInstructions=False,
    ch_ModelInstructions_without_context_without_OutputInstructions=False,

    ch_UserTextInput_with_context_with_OutputFormat=True,
    ch_UserTextInput_with_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_with_context_without_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat=False,
    ch_UserTextInput_without_context_with_OutputFormat_with_example=False,
    ch_UserTextInput_without_context_without_OutputFormat=False
)